# Dataframe Consolidation: PySpark Implementation

**Purpose:** Same three-table architecture as `eda/05_dataframe_consolidation.ipynb` but using PySpark for better performance.

**Performance Goal:** Reduce aggregation time from several minutes (pandas) to seconds (Spark)

**Tables:**
1. `dim_districts` - Geographic dimension
2. `fact_monthly_climate` - Monthly weather + vegetation  
3. `fact_seasonal_crop` - Seasonal crop yields + aggregated weather features

**Key Optimization:** Replace row-by-row pandas `.apply()` with Spark SQL joins and window functions

## Setup

In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    expr,
    when,
    concat_ws,
    explode,
    array,
    lit,
    sum as _sum,
    mean as _mean,
    max as _max,
    min as _min,
    count,
    year,
    month,
    to_date,
)
from pyspark.sql.types import IntegerType, ArrayType
from pathlib import Path
import time

print("✓ Imports complete")

✓ Imports complete


In [2]:
# Initialize Spark Session
spark = (
    SparkSession.builder.appName("Crop Data Consolidation")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 15:40:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.0
Spark UI: http://llp6dd291q4.dhcp.ti.com:4040


In [3]:
# Paths
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

data_path = project_root / "data" / "processed"

print(f"Project root: {project_root}")
print(f"Data path: {data_path}")

Project root: /Users/a0488240/Documents/Learning/MTech/project/dsba2026
Data path: /Users/a0488240/Documents/Learning/MTech/project/dsba2026/data/processed


## Load Source Data

In [4]:
# Load from pandas pickle files and convert to Spark
import pandas as pd

print("Loading from pickle files...\n")

# Load pandas dataframes
print("1. Loading crop_yield...")
crop_yield_pdf = pd.read_pickle(data_path / "crop_yield.pkl")
print(f"   Loaded: {len(crop_yield_pdf):,} rows")

print("2. Loading geography...")
geography_pdf = pd.read_pickle(data_path / "geography.pkl")
print(f"   Loaded: {len(geography_pdf):,} rows")

print("3. Loading precipitation...")
precipitation_pdf = pd.read_pickle(data_path / "precipitation.pkl")
print(f"   Loaded: {len(precipitation_pdf):,} rows")

print("4. Loading vegetation...")
vegetation_pdf = pd.read_pickle(data_path / "vegetation.pkl")
print(f"   Loaded: {len(vegetation_pdf):,} rows")

print("\nConverting to Spark DataFrames...")
# Convert to Spark DataFrames
crop_yield_sdf = spark.createDataFrame(crop_yield_pdf)
geography_sdf = spark.createDataFrame(geography_pdf)
precipitation_sdf = spark.createDataFrame(precipitation_pdf)
vegetation_sdf = spark.createDataFrame(vegetation_pdf)

print("✓ All source data loaded and converted to Spark")

Loading from pickle files...

1. Loading crop_yield...
   Loaded: 93,262 rows
2. Loading geography...
   Loaded: 515 rows
3. Loading precipitation...
   Loaded: 50,040 rows
4. Loading vegetation...
   Loaded: 124,189 rows

Converting to Spark DataFrames...
✓ All source data loaded and converted to Spark


## Data Preparation

In [5]:
# Prepare data
print("Preparing data...\n")

# Extract integer year from datetime
crop_yield_sdf = crop_yield_sdf.withColumn("crop_year", year(col("year")))

# Ensure precipitation and vegetation have integer year/month
precipitation_sdf = precipitation_sdf.withColumn(
    "year", col("year").cast(IntegerType())
)
precipitation_sdf = precipitation_sdf.withColumn(
    "month", col("month").cast(IntegerType())
)

vegetation_sdf = vegetation_sdf.withColumn("year", col("year").cast(IntegerType()))
vegetation_sdf = vegetation_sdf.withColumn("month", col("month").cast(IntegerType()))

print("✓ Data preparation complete")

Preparing data...

✓ Data preparation complete


In [6]:
# Inspect schemas
print("Source Data Schemas:")
print("=" * 80)

print("\nCrop Yield:")
crop_yield_sdf.printSchema()

print("\nPrecipitation:")
precipitation_sdf.printSchema()

print("\nVegetation:")
vegetation_sdf.printSchema()

print("\nGeography:")
geography_sdf.printSchema()

Source Data Schemas:

Crop Yield:
root
 |-- year: timestamp (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- crop_name: string (nullable = true)
 |-- crop_type: string (nullable = true)
 |-- season: string (nullable = true)
 |-- area: double (nullable = true)
 |-- production: double (nullable = true)
 |-- yield: double (nullable = true)
 |-- area_unit: string (nullable = true)
 |-- production_unit: string (nullable = true)
 |-- yield_unit: string (nullable = true)
 |-- season_start_month: long (nullable = true)
 |-- season_end_month: long (nullable = true)
 |-- season_months: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- crop_year: integer (nullable = true)


Precipitation:
root
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- precipitation_mm: double (nullable = true)


Vegetation:
root
 |-- district

In [7]:
# Inspect schemas
print("Source Data Schemas:")
print("=" * 80)

print("\nCrop Yield:")
crop_yield_sdf.printSchema()

print("\nPrecipitation:")
precipitation_sdf.printSchema()

print("\nVegetation:")
vegetation_sdf.printSchema()

print("\nGeography:")
geography_sdf.printSchema()

Source Data Schemas:

Crop Yield:
root
 |-- year: timestamp (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- crop_name: string (nullable = true)
 |-- crop_type: string (nullable = true)
 |-- season: string (nullable = true)
 |-- area: double (nullable = true)
 |-- production: double (nullable = true)
 |-- yield: double (nullable = true)
 |-- area_unit: string (nullable = true)
 |-- production_unit: string (nullable = true)
 |-- yield_unit: string (nullable = true)
 |-- season_start_month: long (nullable = true)
 |-- season_end_month: long (nullable = true)
 |-- season_months: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- crop_year: integer (nullable = true)


Precipitation:
root
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- precipitation_mm: double (nullable = true)


Vegetation:
root
 |-- district

## Table 1: Create District Dimension

In [8]:
print("Creating dim_districts...\n")

# Geography is already aggregated at district level
dim_districts = geography_sdf.withColumn(
    "district_id", concat_ws("_", col("state"), col("district"))
).select("district_id", "state", "district", "centroid_lat", "centroid_lon")

dim_districts.cache()
district_count = dim_districts.count()

print(f"✓ dim_districts created: {district_count:,} rows")
print("\nSample:")
dim_districts.show(5, truncate=False)

Creating dim_districts...



✓ dim_districts created: 515 rows

Sample:
+----------------------------+--------------+-------------+------------------+-----------------+
|district_id                 |state         |district     |centroid_lat      |centroid_lon     |
+----------------------------+--------------+-------------+------------------+-----------------+
|Andhra Pradesh_Adilabad     |Andhra Pradesh|Adilabad     |19.28451395007342 |78.81321233480176|
|Andhra Pradesh_Anantapur    |Andhra Pradesh|Anantapur    |14.312065561843168|77.46015844785772|
|Andhra Pradesh_Chittoor     |Andhra Pradesh|Chittoor     |13.331092630676949|78.92763898886032|
|Andhra Pradesh_Cuddapah     |Andhra Pradesh|Cuddapah     |14.490289134125637|78.69613870967741|
|Andhra Pradesh_East Godavari|Andhra Pradesh|East Godavari|16.782718320784852|82.24320747205111|
+----------------------------+--------------+-------------+------------------+-----------------+
only showing top 5 rows



## Table 2: Create Monthly Climate Table

In [9]:
print("Creating fact_monthly_climate...\n")

# Inner join precipitation and vegetation
fact_monthly_climate = precipitation_sdf.join(
    vegetation_sdf, on=["state", "district", "year", "month"], how="inner"
)

# Add date column
fact_monthly_climate = fact_monthly_climate.withColumn(
    "date", to_date(concat_ws("-", col("year"), col("month"), lit(1)))
)

# Reorder columns
fact_monthly_climate = fact_monthly_climate.select(
    "state",
    "district",
    "year",
    "month",
    "date",
    "precipitation_mm",
    "raw_ndvi_mean",
    "crop_area_km2",
    "valid_crop_area_km2",
    "valid_crop_fraction",
    "quality_status",
)

fact_monthly_climate.cache()
monthly_count = fact_monthly_climate.count()

print(f"✓ fact_monthly_climate created: {monthly_count:,} rows")
print("\nSample:")
fact_monthly_climate.show(10, truncate=False)

Creating fact_monthly_climate...



✓ fact_monthly_climate created: 94,141 rows

Sample:
+--------------+---------+----+-----+----------+------------------+------------------+------------------+-------------------+-------------------+--------------+
|state         |district |year|month|date      |precipitation_mm  |raw_ndvi_mean     |crop_area_km2     |valid_crop_area_km2|valid_crop_fraction|quality_status|
+--------------+---------+----+-----+----------+------------------+------------------+------------------+-------------------+-------------------+--------------+
|Andhra Pradesh|Anantapur|2026|2    |2026-02-01|1.05              |0.381474035533215 |16819.558363946613|13105.797184834051 |0.7791998399272388 |valid         |
|Andhra Pradesh|Anantapur|2026|2    |2026-02-01|1.05              |0.3518423004995761|16819.55836394661 |12833.719044028076 |0.7630235447523795 |valid         |
|Andhra Pradesh|Chittoor |2018|4    |2018-04-01|15.629999999999999|0.3699761861352835|12845.269378576457|8559.94020351731   |0.666388531936411

## Table 3: Create Seasonal Crop Table (Spark Optimized)

**Key Optimization:** Instead of row-by-row aggregation, we:
1. **Explode** season_months array into individual month rows
2. **Add year offset** for months 1-5 (year-wrapping)
3. **Join** with monthly climate data
4. **Group by** season and aggregate

This transforms the problem into set operations that Spark can parallelize.

In [10]:
print("Creating fact_seasonal_crop...")
print("Using Spark-optimized explode + join + groupBy approach...\n")

start_time = time.time()

# Step 1: Explode season_months array
# Each crop-season row becomes multiple rows (one per month in season)
crop_exploded = crop_yield_sdf.withColumn("month", explode(col("season_months")))

# Step 2: Handle year-wrapping
# Months >= 6 use crop_year, months <= 5 use crop_year+1
crop_exploded = crop_exploded.withColumn(
    "weather_year",
    when(col("month") >= 6, col("crop_year")).otherwise(col("crop_year") + 1),
)

print("Step 1: Exploded season_months and added year offset")
print(f"  Exploded rows: {crop_exploded.count():,}")

# Step 3: Join with precipitation
crop_with_precip = crop_exploded.join(
    precipitation_sdf,
    (crop_exploded["state"] == precipitation_sdf["state"])
    & (crop_exploded["district"] == precipitation_sdf["district"])
    & (crop_exploded["weather_year"] == precipitation_sdf["year"])
    & (crop_exploded["month"] == precipitation_sdf["month"]),
    how="left",
).select(crop_exploded["*"], precipitation_sdf["precipitation_mm"])

print("Step 2: Joined with precipitation")

# Step 4: Join with vegetation
crop_with_weather = crop_with_precip.join(
    vegetation_sdf,
    (crop_with_precip["state"] == vegetation_sdf["state"])
    & (crop_with_precip["district"] == vegetation_sdf["district"])
    & (crop_with_precip["weather_year"] == vegetation_sdf["year"])
    & (crop_with_precip["month"] == vegetation_sdf["month"]),
    how="left",
).select(crop_with_precip["*"], vegetation_sdf["raw_ndvi_mean"])

print("Step 3: Joined with vegetation")

# Step 5: Aggregate back to seasonal level
fact_seasonal_crop = crop_with_weather.groupBy(
    "state",
    "district",
    "crop_year",
    "crop_name",
    "crop_type",
    "season",
    "area",
    "production",
    "yield",
    "season_start_month",
    "season_end_month",
).agg(
    # Precipitation features
    _sum("precipitation_mm").alias("seasonal_precip_total"),
    _mean("precipitation_mm").alias("seasonal_precip_avg"),
    _max("precipitation_mm").alias("seasonal_precip_max"),
    _sum(when(col("precipitation_mm") > 2.5, 1).otherwise(0)).alias("rainy_days"),
    # Vegetation features
    _mean("raw_ndvi_mean").alias("seasonal_ndvi_mean"),
    _min("raw_ndvi_mean").alias("seasonal_ndvi_min"),
    _max("raw_ndvi_mean").alias("seasonal_ndvi_max"),
)

print("Step 4: Aggregated to seasonal level")

# Step 6: Add phase-specific NDVI (early, mid, late)
# This requires a more complex approach - for now, we'll use seasonal mean as placeholder
# TODO: Implement phase-specific NDVI using window functions
fact_seasonal_crop = (
    fact_seasonal_crop.withColumn("early_season_ndvi", col("seasonal_ndvi_mean"))
    .withColumn("mid_season_ndvi", col("seasonal_ndvi_mean"))
    .withColumn("late_season_ndvi", col("seasonal_ndvi_mean"))
)

# Step 7: Add geographic features
fact_seasonal_crop = fact_seasonal_crop.join(
    dim_districts.select("state", "district", "centroid_lat", "centroid_lon"),
    on=["state", "district"],
    how="left",
)

# Reorder columns
fact_seasonal_crop = fact_seasonal_crop.select(
    "state",
    "district",
    "crop_year",
    "crop_name",
    "crop_type",
    "season",
    "area",
    "production",
    "yield",
    "season_start_month",
    "season_end_month",
    "seasonal_precip_total",
    "seasonal_precip_avg",
    "seasonal_precip_max",
    "rainy_days",
    "seasonal_ndvi_mean",
    "seasonal_ndvi_min",
    "seasonal_ndvi_max",
    "early_season_ndvi",
    "mid_season_ndvi",
    "late_season_ndvi",
    "centroid_lat",
    "centroid_lon",
)

fact_seasonal_crop.cache()
seasonal_count = fact_seasonal_crop.count()

elapsed = time.time() - start_time

print(f"\n✓ fact_seasonal_crop created: {seasonal_count:,} rows")
print(f"⚡ Aggregation complete in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print("\nSample:")
fact_seasonal_crop.show(5, truncate=False)

Creating fact_seasonal_crop...
Using Spark-optimized explode + join + groupBy approach...

Step 1: Exploded season_months and added year offset
  Exploded rows: 613,255
Step 2: Joined with precipitation
Step 3: Joined with vegetation
Step 4: Aggregated to seasonal level



✓ fact_seasonal_crop created: 93,262 rows
⚡ Aggregation complete in 11.8 seconds (0.2 minutes)

Sample:
+--------------+----------------+---------+---------------------+---------+------+------+----------+-----+------------------+----------------+---------------------+-------------------+-------------------+----------+-------------------+------------------+------------------+-------------------+-------------------+-------------------+------------------+-----------------+
|state         |district        |crop_year|crop_name            |crop_type|season|area  |production|yield|season_start_month|season_end_month|seasonal_precip_total|seasonal_precip_avg|seasonal_precip_max|rainy_days|seasonal_ndvi_mean |seasonal_ndvi_min |seasonal_ndvi_max |early_season_ndvi  |mid_season_ndvi    |late_season_ndvi   |centroid_lat      |centroid_lon     |
+--------------+----------------+---------+---------------------+---------+------+------+----------+-----+------------------+----------------+-----------

## Verification Steps

In [11]:
print("VERIFICATION CHECKS")
print("=" * 80)

# 1. Check row counts
print("\n1. Row Counts:")
print(f"   dim_districts:         {district_count:>8,} rows (expected ~515)")
print(f"   fact_monthly_climate:  {monthly_count:>8,} rows (expected ~55,000)")
print(f"   fact_seasonal_crop:    {seasonal_count:>8,} rows (expected ~357,000)")

# 2. Check for duplicates
print("\n2. Duplicate Check:")
dim_dups = (
    dim_districts.groupBy("state", "district").count().filter(col("count") > 1).count()
)
monthly_dups = (
    fact_monthly_climate.groupBy("state", "district", "year", "month")
    .count()
    .filter(col("count") > 1)
    .count()
)
seasonal_dups = (
    fact_seasonal_crop.groupBy("state", "district", "crop_year", "crop_name", "season")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"   dim_districts duplicates:        {dim_dups} (must be 0)")
print(f"   fact_monthly_climate duplicates: {monthly_dups} (must be 0)")
print(f"   fact_seasonal_crop duplicates:   {seasonal_dups} (must be 0)")

if dim_dups == 0 and monthly_dups == 0 and seasonal_dups == 0:
    print("   ✓ No duplicates found")
else:
    print("   ❌ Duplicates detected!")

# 3. Check coverage
print("\n3. Weather Data Coverage in Seasonal Crop:")
total_rows = seasonal_count
precip_coverage = (
    fact_seasonal_crop.filter(col("seasonal_precip_total").isNotNull()).count()
    / total_rows
    * 100
)
ndvi_coverage = (
    fact_seasonal_crop.filter(col("seasonal_ndvi_mean").isNotNull()).count()
    / total_rows
    * 100
)
print(f"   Precipitation coverage: {precip_coverage:>5.1f}%")
print(f"   NDVI coverage:          {ndvi_coverage:>5.1f}%")

# 4. Sample Rabi validation
print("\n4. Rabi Year-Wrapping Validation:")
sample_rabi = fact_seasonal_crop.filter(
    (col("season") == "Rabi")
    & (col("crop_year") == 2020)
    & col("seasonal_precip_total").isNotNull()
).limit(1)

if sample_rabi.count() > 0:
    sample_rabi.select("state", "district", "seasonal_precip_total").show(
        truncate=False
    )
    print("   ✓ Year-wrapping appears correct")
else:
    print("   ⚠️  No Rabi 2020 records with precipitation found")

print("\n" + "=" * 80)
print("✓ Verification complete!")

VERIFICATION CHECKS

1. Row Counts:
   dim_districts:              515 rows (expected ~515)
   fact_monthly_climate:    94,141 rows (expected ~55,000)
   fact_seasonal_crop:      93,262 rows (expected ~357,000)

2. Duplicate Check:


   dim_districts duplicates:        0 (must be 0)
   fact_monthly_climate duplicates: 38969 (must be 0)
   fact_seasonal_crop duplicates:   0 (must be 0)
   ❌ Duplicates detected!

3. Weather Data Coverage in Seasonal Crop:
   Precipitation coverage:  93.6%
   NDVI coverage:           99.0%

4. Rabi Year-Wrapping Validation:
+--------------+----------+---------------------+
|state         |district  |seasonal_precip_total|
+--------------+----------+---------------------+
|Andhra Pradesh|Srikakulam|83.97999999999998    |
+--------------+----------+---------------------+

   ✓ Year-wrapping appears correct

✓ Verification complete!


## Performance Comparison

In [12]:
print("PERFORMANCE COMPARISON")
print("=" * 80)
print("\nPandas (eda/05_dataframe_consolidation.ipynb):")
print("  - Method: Row-by-row .apply() with lambda function")
print("  - Time: Several minutes (2-5 minutes typical)")
print("  - Memory: Single-threaded, ~2-4GB")

print("\nPySpark (this notebook):")
print("  - Method: Explode + Join + GroupBy (parallel)")
print(f"  - Time: {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")
print("  - Memory: Distributed across cores, better scalability")

if elapsed < 120:  # Less than 2 minutes
    speedup = 180 / elapsed  # Assume pandas took ~3 minutes
    print(f"\n⚡ Estimated speedup: {speedup:.1f}x faster than pandas!")
print("\n" + "=" * 80)

PERFORMANCE COMPARISON

Pandas (eda/05_dataframe_consolidation.ipynb):
  - Method: Row-by-row .apply() with lambda function
  - Time: Several minutes (2-5 minutes typical)
  - Memory: Single-threaded, ~2-4GB

PySpark (this notebook):
  - Method: Explode + Join + GroupBy (parallel)
  - Time: 11.8 seconds (0.20 minutes)
  - Memory: Distributed across cores, better scalability

⚡ Estimated speedup: 15.2x faster than pandas!



## Save Three Tables

In [13]:
print("Saving tables...\n")

output_path = str(data_path)

# Save as Parquet (Spark native format)
print("Saving parquet files...")
dim_districts.write.mode("overwrite").parquet(
    f"{output_path}/dim_districts_spark.parquet"
)
print("  ✓ dim_districts_spark.parquet")

fact_monthly_climate.write.mode("overwrite").parquet(
    f"{output_path}/fact_monthly_climate_spark.parquet"
)
print("  ✓ fact_monthly_climate_spark.parquet")

fact_seasonal_crop.write.mode("overwrite").parquet(
    f"{output_path}/fact_seasonal_crop_spark.parquet"
)
print("  ✓ fact_seasonal_crop_spark.parquet")

print("\n" + "=" * 80)
print("✓ All tables saved successfully!")
print(f"\nLocation: {output_path}")
print("\nNote: Files saved with '_spark' suffix to distinguish from pandas versions")

Saving tables...

Saving parquet files...


  ✓ dim_districts_spark.parquet


  ✓ fact_monthly_climate_spark.parquet


  ✓ fact_seasonal_crop_spark.parquet

✓ All tables saved successfully!

Location: /Users/a0488240/Documents/Learning/MTech/project/dsba2026/data/processed

Note: Files saved with '_spark' suffix to distinguish from pandas versions


## Summary

In [14]:
print("SUMMARY")
print("=" * 80)
print("\nThree tables created using PySpark:")
print(f"\n1. dim_districts ({district_count:,} rows)")
print("   Purpose: Geographic dimension with district centroids")
print("   Key: state, district")

print(f"\n2. fact_monthly_climate ({monthly_count:,} rows)")
print("   Purpose: Monthly weather + vegetation data")
print("   Key: state, district, year, month")

print(f"\n3. fact_seasonal_crop ({seasonal_count:,} rows)")
print("   Purpose: Seasonal crop yields + aggregated weather features")
print("   Key: state, district, crop_year, crop_name, season")

print("\n" + "=" * 80)
print("\nKEY OPTIMIZATIONS:")
print("-" * 80)
print("\n1. Explode-Join-Aggregate Pattern")
print("   - Replaced row-by-row pandas .apply() with set operations")
print("   - Enables parallel processing across Spark executors")

print("\n2. Year-Wrapping Logic")
print("   - Used when() expression instead of Python conditionals")
print("   - Vectorized operation across all rows simultaneously")

print("\n3. Caching Strategy")
print("   - Cached intermediate results to avoid recomputation")
print("   - Particularly important for exploded dataset")

print("\n" + "=" * 80)
print(
    "\n⚡ PERFORMANCE WIN: PySpark significantly faster for large-scale aggregations!"
)

SUMMARY

Three tables created using PySpark:

1. dim_districts (515 rows)
   Purpose: Geographic dimension with district centroids
   Key: state, district

2. fact_monthly_climate (94,141 rows)
   Purpose: Monthly weather + vegetation data
   Key: state, district, year, month

3. fact_seasonal_crop (93,262 rows)
   Purpose: Seasonal crop yields + aggregated weather features
   Key: state, district, crop_year, crop_name, season


KEY OPTIMIZATIONS:
--------------------------------------------------------------------------------

1. Explode-Join-Aggregate Pattern
   - Replaced row-by-row pandas .apply() with set operations
   - Enables parallel processing across Spark executors

2. Year-Wrapping Logic
   - Used when() expression instead of Python conditionals
   - Vectorized operation across all rows simultaneously

3. Caching Strategy
   - Cached intermediate results to avoid recomputation
   - Particularly important for exploded dataset


⚡ PERFORMANCE WIN: PySpark significantly faster

## Cleanup

In [15]:
# Unpersist cached dataframes
dim_districts.unpersist()
fact_monthly_climate.unpersist()
fact_seasonal_crop.unpersist()

# Stop Spark session
# spark.stop()
# print("✓ Spark session stopped")

DataFrame[state: string, district: string, crop_year: int, crop_name: string, crop_type: string, season: string, area: double, production: double, yield: double, season_start_month: bigint, season_end_month: bigint, seasonal_precip_total: double, seasonal_precip_avg: double, seasonal_precip_max: double, rainy_days: bigint, seasonal_ndvi_mean: double, seasonal_ndvi_min: double, seasonal_ndvi_max: double, early_season_ndvi: double, mid_season_ndvi: double, late_season_ndvi: double, centroid_lat: double, centroid_lon: double]

## Next Steps

1. **Production ETL:** Use this Spark implementation for production pipelines
2. **Phase-specific NDVI:** Implement early/mid/late NDVI using window functions
3. **Partitioning:** Add year-based partitioning for faster queries
4. **Data Quality:** Add Great Expectations validation suite

See CLAUDE.md for production deployment guidelines.